# Activation Functions: why a network needs them

We reuse the plain MLP + Fashion-MNIST training loop (no convolutions) and change **only the activation function** between the layers.

**Plan**
1. See what activations look like (and their gradients, via autograd)
2. Build the MLP so the activation is a parameter
3. Show that **without** an activation, extra layers add nothing (they collapse into one)
4. **Predict first**, then train with: no activation, ReLU, and Sigmoid
5. Read the results: underfitting, good fit, slow learning
6. Bonus: more activations (Tanh, LeakyReLU, GELU), and a look at *why* Sigmoid learns slowly (vanishing gradients)

## 0. Setup

In [ ]:
import math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. What do activation functions look like?

An activation is a simple function applied to each neuron's output, `a = f(z)`. The shape of `f` matters, and so does its **slope** (derivative), because the slope is what the gradient flows through during backprop. We get the slopes from autograd: no calculus by hand.

In [ ]:
z = torch.linspace(-6, 6, 400, requires_grad=True)
acts = {'none (Identity)': nn.Identity(), 'ReLU': nn.ReLU(), 'Sigmoid': nn.Sigmoid(), 'Tanh': nn.Tanh()}

fig, axes = plt.subplots(2, 4, figsize=(15, 5.5), sharex=True)
for i, (name, f) in enumerate(acts.items()):
    y = f(z)
    grad, = torch.autograd.grad(y.sum(), z)          # dy/dz at every point
    axes[0, i].plot(z.detach(), y.detach()); axes[0, i].set_title(name)
    axes[1, i].plot(z.detach(), grad, color='C1');   axes[1, i].set_ylim(-0.1, 1.2)
axes[0, 0].set_ylabel('f(z)'); axes[1, 0].set_ylabel("slope  f'(z)")
for a in axes[1]: a.set_xlabel('z')
plt.tight_layout(); plt.show()

Things to notice in the bottom row (the slopes):
- **Identity**: slope is always 1. It does nothing, which is exactly the problem.
- **ReLU**: slope is 0 for negative inputs and 1 for positive ones. Gradients pass through unchanged.
- **Sigmoid**: slope is at most **0.25**, and nearly 0 at both ends. Every layer of sigmoids shrinks the gradient by at least 4x.
- **Tanh**: slope is at most 1 but also goes flat at both ends.

## 2. Data

In [ ]:
mean, std = 0.2860, 0.3530
tr = datasets.FashionMNIST('data', train=True,  download=True)
te = datasets.FashionMNIST('data', train=False, download=True)

def to_tensors(ds):
    x = (ds.data.float() / 255 - mean) / std
    return x.unsqueeze(1), ds.targets

Xtr, ytr = to_tensors(tr)
Xte, yte = to_tensors(te)

train_loader      = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
train_eval_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=1000)   # for measuring train accuracy
test_loader       = DataLoader(TensorDataset(Xte, yte), batch_size=1000)
print(Xtr.shape, 'train |', Xte.shape, 'test')

## 3. The model: activation as a parameter

Same plain MLP: `784 -> 128 -> 64 -> 10`. The only change is that the activation *class* is passed in, so we can swap `nn.ReLU`, `nn.Sigmoid`, `nn.Identity` (does nothing), and so on.

In [ ]:
class MLP(nn.Module):
    def __init__(self, act=nn.ReLU, hidden=(128, 64)):
        super().__init__()
        layers, d = [nn.Flatten()], 784
        for h in hidden:
            layers += [nn.Linear(d, h), act()]       # linear layer followed by the activation
            d = h
        layers.append(nn.Linear(d, 10))              # output layer: raw logits, no activation
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

print(MLP(nn.ReLU))
print()
print(MLP(nn.Identity))

## 4. Without an activation, extra layers are useless

If every layer is linear, stacking them gives another linear function:

```
W3 (W2 (W1 x + b1) + b2) + b3  =  (W3 W2 W1) x + (W3 W2 b1 + W3 b2 + b3)  =  W x + b
```

So the 3-layer network with `nn.Identity` is exactly a **single** linear layer, however wide or deep it is. Check it with the real model: collapse the three layers into one matrix and compare the outputs.

In [ ]:
torch.manual_seed(0)
m = MLP(nn.Identity).eval()
L1, L2, L3 = [l for l in m.net if isinstance(l, nn.Linear)]

# collapse the three layers into ONE weight matrix and ONE bias vector
W = L3.weight @ L2.weight @ L1.weight                                # (10, 784)
b = L3.weight @ (L2.weight @ L1.bias + L2.bias) + L3.bias            # (10,)

x = Xte[:5]
with torch.no_grad():
    out_deep   = m(x)                                                # the 3-layer network
    out_single = x.flatten(1) @ W.T + b                              # one linear layer
print('max |difference| between the two outputs:', (out_deep - out_single).abs().max().item())
print('parameters in the 3-layer net :', sum(p.numel() for p in m.parameters()))
print('parameters in the single layer:', W.numel() + b.numel())

The outputs match (up to rounding error). The 3-layer network has about 14x more parameters, but can represent **no more** than a plain linear classifier (like logistic regression). The activation is what lets layers do something different from each other.

## 5. Training function

The same 5-step loop as before, wrapped in `run(act)`. Every run uses the same seed, so the starting weights and batch order are identical, and the *only* difference is the activation. We also record **train** accuracy, because the gap between train and test accuracy tells us whether the model is under- or over-fitting.

In [ ]:
criterion = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total / n, correct / n

def run(act, epochs=10, lr=0.01, momentum=0.9, seed=0, hidden=(128, 64), verbose=True, name=None):
    torch.manual_seed(seed)                       # same init + same batch order for every run
    model = MLP(act, hidden).to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    h = dict(step_loss=[], train_loss=[], train_acc=[], test_loss=[], test_acc=[])
    name = name or act.__name__

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()                 # 1. clear gradients
            loss = criterion(model(x), y)         # 2-3. forward + loss
            loss.backward()                       # 4. backward
            optimizer.step()                      # 5. update
            h['step_loss'].append(loss.item())
        trl, tra = evaluate(model, train_eval_loader)
        tel, tea = evaluate(model, test_loader)
        for k, v in zip(['train_loss', 'train_acc', 'test_loss', 'test_acc'], [trl, tra, tel, tea]):
            h[k].append(v)
        if verbose:
            print(f'  {name:10s} epoch {epoch:2d}  train_loss={trl:.4f} train_acc={tra:.4f}  test_loss={tel:.4f} test_acc={tea:.4f}')
    return h

## 6. Predict first

Same setting for every run: SGD, lr 0.01, momentum 0.9, 10 epochs, batch size 64. Chance level is 10%.

| Activation | Test accuracy after 10 epochs (guess) | Train accuracy vs test accuracy | Learning speed (fast/slow) |
|---|---|---|---|
| none (Identity) | | | |
| ReLU | | | |
| Sigmoid | | | |

## 7. The experiment: none vs ReLU vs Sigmoid

In [ ]:
EPOCHS = 10
main_acts = {'none (Identity)': nn.Identity, 'ReLU': nn.ReLU, 'Sigmoid': nn.Sigmoid}

results = {}
for name, act in main_acts.items():
    t0 = time.time()
    results[name] = run(act, epochs=EPOCHS, verbose=False, name=name)
    r = results[name]
    print(f'{name:16s} train_acc={r["train_acc"][-1]:.4f}  test_acc={r["test_acc"][-1]:.4f}  ({time.time()-t0:.0f}s)')

### 7a. Learning curves

In [ ]:
def plot_curves(results, title=''):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for name, r in results.items():
        e = range(1, len(r['train_loss']) + 1)
        ax[0].plot(e, r['train_loss'], marker='o', ms=3, label=name)
        ax[1].plot(e, r['train_acc'],  marker='o', ms=3, label=name)
        ax[2].plot(e, r['test_acc'],   marker='o', ms=3, label=name)
    ax[0].set_title('train loss'); ax[1].set_title('train accuracy'); ax[2].set_title('test accuracy')
    ax[1].set_ylim(0.6, 1); ax[2].set_ylim(0.6, 1)
    for a in ax: a.set_xlabel('epoch')
    ax[0].legend(); fig.suptitle(title); plt.tight_layout(); plt.show()

plot_curves(results, 'Same data, same lr, same init: only the activation changes')

### 7b. Summary

In [ ]:
def summary(results):
    rows = []
    for name, r in results.items():
        rows.append({'activation': name,
                     'train_loss': r['train_loss'][-1],
                     'train_acc': r['train_acc'][-1],
                     'test_acc': r['test_acc'][-1],
                     'train-test gap': r['train_acc'][-1] - r['test_acc'][-1],
                     'test_acc after epoch 1': r['test_acc'][0]})
    return pd.DataFrame(rows).set_index('activation').round(4)

display(summary(results))

## 8. What to look for

- **No activation:** accuracy rises quickly and then **plateaus** (typically in the mid-80s). Train and test accuracy stay close together: the model is **underfitting**. It is limited by what a linear model can express, and more training, more width or more layers will not fix it.
- **ReLU:** a few points higher, with train accuracy pulling ahead of test accuracy. The model now has the flexibility to fit the data (and starts to overfit if trained long enough).
- **Sigmoid:** learns, but **more slowly**. After epoch 1 it typically trails ReLU clearly, and it may or may not catch up within 10 epochs. The slope of the sigmoid is at most 0.25 and flat at the extremes, so gradients are weak. Section 10 measures this.

The lesson: the activation is what turns a stack of matrix multiplications into a model that can learn non-linear patterns, and the *choice* of activation affects how fast it learns.

## 9. Bonus A: more activations

Add Tanh, LeakyReLU and GELU to the picture. ReLU-family activations (ReLU, LeakyReLU, GELU) are normally within a fraction of a point of each other here. Tanh sits between ReLU and Sigmoid.

In [ ]:
more_acts = {'Tanh': nn.Tanh, 'LeakyReLU': nn.LeakyReLU, 'GELU': nn.GELU}
all_results = dict(results)
for name, act in more_acts.items():
    all_results[name] = run(act, epochs=EPOCHS, verbose=False, name=name)
    print(f'{name:10s} train_acc={all_results[name]["train_acc"][-1]:.4f}  test_acc={all_results[name]["test_acc"][-1]:.4f}')

plot_curves(all_results, 'All activations')
display(summary(all_results))

## 10. Bonus B: why is Sigmoid slow? Vanishing gradients, measured

During backprop the gradient is multiplied by the activation's slope at every layer. For Sigmoid that slope is at most 0.25, so after several layers the gradient reaching the **early** layers is tiny, and those layers barely learn.

Let's see it directly. We build a deeper network (8 hidden layers of width 64), run **one** backward pass on one batch, and plot the gradient size at each layer from input to output. No training is needed. The effect is much easier to see in a deep net than in our 2-hidden-layer one.

In [ ]:
def grad_per_layer(act, depth=8, width=64, seed=0):
    torch.manual_seed(seed)
    model = MLP(act, hidden=(width,) * depth).to(device)
    x, y = next(iter(train_loader))
    x, y = x.to(device), y.to(device)
    model.zero_grad()
    criterion(model(x), y).backward()
    # gradient norm of each Linear layer's weight, from the input side to the output side
    return [l.weight.grad.norm().item() for l in model.net if isinstance(l, nn.Linear)]

fig, ax = plt.subplots(figsize=(8, 4.2))
for name, act in {'none (Identity)': nn.Identity, 'ReLU': nn.ReLU, 'Tanh': nn.Tanh, 'Sigmoid': nn.Sigmoid}.items():
    g = grad_per_layer(act)
    ax.plot(range(1, len(g) + 1), g, marker='o', label=name)
    print(f'{name:16s} first-layer grad / last-layer grad = {g[0] / g[-1]:.2e}')
ax.set_yscale('log'); ax.set_xlabel('layer (1 = closest to the input, 9 = output)')
ax.set_ylabel('gradient norm of layer weights (log scale)')
ax.set_title('Gradient size at each layer, deep net (8 hidden layers), at initialization')
ax.legend(); plt.show()

Read the plot from right (output layer) to left (input layers). The gradient shrinks as it travels backward, and it typically shrinks **much faster for Sigmoid** than for the others. The early layers of a deep sigmoid network get almost no learning signal. This is the **vanishing gradient problem**, and a major reason ReLU became the default.

(All of these shrink somewhat with PyTorch's default initialization, so compare the *relative* decay between activations, not the absolute values.)

## Takeaways

1. **No activation = linear model**, no matter how many layers. It plateaus early and underfits.
2. **ReLU** (and relatives) give the network non-linear power, and gradients flow through easily (slope 1 for positive inputs). It is the default for good reason.
3. **Sigmoid and Tanh** work, but their flat tails shrink gradients, which slows learning, and it gets worse the deeper the network is.
4. Compare activations at a fixed seed, lr and data, and look at **both** the train and test curves to separate underfitting from overfitting.

## Exercises
1. Make the no-activation network much wider (`hidden=(1024, 1024)`). Does accuracy improve? Why not?
2. Train Sigmoid for 30 epochs. Does it catch up to ReLU?
3. Train Sigmoid with `lr=0.1`. Does a higher learning rate compensate for the weak gradients?
4. Try `nn.ELU` and `nn.SiLU`. Where do they land?
5. In section 10, change `depth` to 3, 8, and 16. How does the first-layer / last-layer gradient ratio change for each activation?
6. Put the activation only after the first hidden layer (none after the second). How many non-linearities does the network need to get most of the benefit?